In [38]:
import tensorflow as tf
import numpy as np
import pandas as pd
import time
from tensorflow.keras.models import Sequential,Model
from numpy.lib.recfunctions import join_by
from sympy import sequence



In [39]:
text=open('khayyam.txt','rb').read().decode('utf-8')
text[:10]

'|زاهدی را '

In [40]:
vocabolaries=sorted(set(text))
vocabolaries[:10]

['\n', '\r', ' ', '|', '،', 'آ', 'ا', 'ب', 'ت', 'ث']

In [41]:
len(vocabolaries)

37

In [42]:
char2index={u:i for i,u in enumerate(vocabolaries) }
index2char=np.array(vocabolaries)
char2index

{'\n': 0,
 '\r': 1,
 ' ': 2,
 '|': 3,
 '،': 4,
 'آ': 5,
 'ا': 6,
 'ب': 7,
 'ت': 8,
 'ث': 9,
 'ج': 10,
 'ح': 11,
 'خ': 12,
 'د': 13,
 'ذ': 14,
 'ر': 15,
 'ز': 16,
 'س': 17,
 'ش': 18,
 'ص': 19,
 'ض': 20,
 'ع': 21,
 'غ': 22,
 'ف': 23,
 'ق': 24,
 'ل': 25,
 'م': 26,
 'ن': 27,
 'ه': 28,
 'و': 29,
 'ُ': 30,
 'ٔ': 31,
 'پ': 32,
 'چ': 33,
 'ک': 34,
 'گ': 35,
 'ی': 36}

In [43]:
text_as_integer=np.array([char2index[c] for c in text])
text_as_integer

array([ 3, 16,  6, 28, 13, 36,  2, 15,  6,  2, 35, 23,  8,  2, 36,  6, 15,
       36,  2, 13, 15,  2, 21, 26, 25,  1,  0,  3, 34, 26,  2, 35, 15, 36,
        2,  8,  6,  2, 33, 18, 26,  2, 15,  6,  2, 27,  6, 36, 13,  2, 12,
       25, 25,  1,  0,  3, 35, 23,  8,  2, 16,  6, 28, 13,  2,  6, 16,  2,
       13, 29,  2,  7, 36, 15, 29, 27,  2, 27, 36, 17,  8,  2, 11,  6, 25,
        1,  0,  3, 33, 18, 26,  2,  7, 36, 27, 13,  2, 36,  6,  2, 27,  7,
       36, 27, 13,  2,  5, 27,  2, 10, 26,  6, 25,  1,  0,  3, 35, 15,  2,
        7,  7, 36, 27, 13,  2, 27, 29, 15,  2, 11, 24,  2, 12, 29, 13,  2,
       33, 28,  2, 22, 26, 17,  8,  1,  0,  3, 13, 15,  2, 29, 19,  6, 25,
        2, 11, 24,  2, 13, 29,  2, 13, 36, 13, 28,  2, 33, 28,  2, 34, 26,
       17,  8,  1,  0,  3, 29, 15,  2, 27, 12, 29,  6, 28, 13,  2, 13, 36,
       13,  2, 11, 24,  2, 15,  6,  2, 35, 29,  2,  7, 15, 29,  1,  0,  3,
        6, 36, 27,  2, 33, 27, 36, 27,  2, 33, 18, 26,  2, 18, 24, 36,  2,
       35, 29,  2, 34, 29

In [44]:
char_dataset=tf.data.Dataset.from_tensor_slices(text_as_integer)
char_dataset

<_TensorSliceDataset element_spec=TensorSpec(shape=(), dtype=tf.int64, name=None)>

In [45]:
for i in char_dataset:
    print(index2char[i.numpy()])

|
ز
ا
ه
د
ی
 
ر
ا
 
گ
ف
ت
 
ی
ا
ر
ی
 
د
ر
 
ع
م
ل



|
ک
م
 
گ
ر
ی
 
ت
ا
 
چ
ش
م
 
ر
ا
 
ن
ا
ی
د
 
خ
ل
ل



|
گ
ف
ت
 
ز
ا
ه
د
 
ا
ز
 
د
و
 
ب
ی
ر
و
ن
 
ن
ی
س
ت
 
ح
ا
ل



|
چ
ش
م
 
ب
ی
ن
د
 
ی
ا
 
ن
ب
ی
ن
د
 
آ
ن
 
ج
م
ا
ل



|
گ
ر
 
ب
ب
ی
ن
د
 
ن
و
ر
 
ح
ق
 
خ
و
د
 
چ
ه
 
غ
م
س
ت



|
د
ر
 
و
ص
ا
ل
 
ح
ق
 
د
و
 
د
ی
د
ه
 
چ
ه
 
ک
م
س
ت



|
و
ر
 
ن
خ
و
ا
ه
د
 
د
ی
د
 
ح
ق
 
ر
ا
 
گ
و
 
ب
ر
و



|
ا
ی
ن
 
چ
ن
ی
ن
 
چ
ش
م
 
ش
ق
ی
 
گ
و
 
ک
و
ر
 
ش
و



|
غ
م
 
م
خ
و
ر
 
ا
ز
 
د
ی
د
ه
،
 
ک
ا
ن
 
ع
ی
س
ی
 
ت
ر
ا
س
ت



|
چ
پ
 
م
ر
و
 
ت
ا
 
ب
خ
ش
د
ت
 
د
و
 
چ
ش
م
 
ر
ا
س
ت



|
ع
ی
س
ی
 
ر
و
ح
 
ت
و
 
ب
ا
 
ت
و
 
ح
ا
ض
ر
س
ت



|
ن
ص
ر
ت
 
ا
ز
 
و
ی
 
خ
و
ا
ه
 
ک
و
 
خ
و
ش
 
ن
ا
ص
ر
س
ت



|
ل
ی
ک
 
ب
ی
گ
ا
ر
 
ت
ن
 
پ
ر
 
ا
س
ت
خ
و
ا
ن



|
ب
ر
 
د
ل
 
ع
ی
س
ی
 
م
ن
ه
 
ت
و
 
ه
ر
 
ز
م
ا
ن



|
ه
م
چ
و
 
آ
ن
 
ا
ب
ل
ه
 
ک
ه
 
ا
ن
د
ر
 
د
ا
س
ت
ا
ن



|
ذ
ک
ر
 
ا
و
 
ک
ر
د
ی
م
 
ب
ه
ر
 
ر
ا
س
ت
ا
ن



|
ز
ن
د
گ
ی
 
ت
ن
 
م
ج
و
 
ا
ز
 
ع
ی
س
ی
 
ا
ت



|
ک
ا
م
 
ف
ر
ع
و
ن
ی
 
م
خ
و
ا
ه
 

In [46]:
sequence=char_dataset.batch(101,drop_remainder=True)
sequence

<_BatchDataset element_spec=TensorSpec(shape=(101,), dtype=tf.int64, name=None)>

In [47]:
for i in sequence.take(3):
    print('---->',''.join(index2char[i.numpy()]))

----> |زاهدی را گفت یاری در عمل
|کم گری تا چشم را ناید خلل
|گفت زاهد از دو بیرون نیست حال
|چشم بیند یا ن
----> بیند آن جمال
|گر ببیند نور حق خود چه غمست
|در وصال حق دو دیده چه کمست
|ور نخواهد دید حق را گو برو
----> 
|این چنین چشم شقی گو کور شو
|غم مخور از دیده، کان عیسی تراست
|چپ مرو تا بخشدت دو چشم راست
|عیسی ر


In [48]:
def sit(batch):
    input_txt=batch[:-1]
    target_txt=batch[1:]
    return input_txt,target_txt
dataset=sequence.map(sit)

In [49]:
dataset

<_MapDataset element_spec=(TensorSpec(shape=(100,), dtype=tf.int64, name=None), TensorSpec(shape=(100,), dtype=tf.int64, name=None))>

In [50]:
for i in dataset.take(1):
    print(i)

(<tf.Tensor: shape=(100,), dtype=int64, numpy=
array([ 3, 16,  6, 28, 13, 36,  2, 15,  6,  2, 35, 23,  8,  2, 36,  6, 15,
       36,  2, 13, 15,  2, 21, 26, 25,  1,  0,  3, 34, 26,  2, 35, 15, 36,
        2,  8,  6,  2, 33, 18, 26,  2, 15,  6,  2, 27,  6, 36, 13,  2, 12,
       25, 25,  1,  0,  3, 35, 23,  8,  2, 16,  6, 28, 13,  2,  6, 16,  2,
       13, 29,  2,  7, 36, 15, 29, 27,  2, 27, 36, 17,  8,  2, 11,  6, 25,
        1,  0,  3, 33, 18, 26,  2,  7, 36, 27, 13,  2, 36,  6,  2])>, <tf.Tensor: shape=(100,), dtype=int64, numpy=
array([16,  6, 28, 13, 36,  2, 15,  6,  2, 35, 23,  8,  2, 36,  6, 15, 36,
        2, 13, 15,  2, 21, 26, 25,  1,  0,  3, 34, 26,  2, 35, 15, 36,  2,
        8,  6,  2, 33, 18, 26,  2, 15,  6,  2, 27,  6, 36, 13,  2, 12, 25,
       25,  1,  0,  3, 35, 23,  8,  2, 16,  6, 28, 13,  2,  6, 16,  2, 13,
       29,  2,  7, 36, 15, 29, 27,  2, 27, 36, 17,  8,  2, 11,  6, 25,  1,
        0,  3, 33, 18, 26,  2,  7, 36, 27, 13,  2, 36,  6,  2, 27])>)


In [51]:
for i in dataset.take(1):
    print(''.join(index2char[i[0].numpy()]))
    print(''.join(index2char[i[1].numpy()]))


|زاهدی را گفت یاری در عمل
|کم گری تا چشم را ناید خلل
|گفت زاهد از دو بیرون نیست حال
|چشم بیند یا 
زاهدی را گفت یاری در عمل
|کم گری تا چشم را ناید خلل
|گفت زاهد از دو بیرون نیست حال
|چشم بیند یا ن


In [52]:
dataset=dataset.batch(64,drop_remainder=True)
dataset

<_BatchDataset element_spec=(TensorSpec(shape=(64, 100), dtype=tf.int64, name=None), TensorSpec(shape=(64, 100), dtype=tf.int64, name=None))>

In [53]:
vocabulary_size=len(vocabolaries)
embding_dim=25
rnn_units=1024

In [54]:
from tensorflow.keras.layers import Embedding,Dense,GRU
model = Sequential()
model.add(Embedding(input_dim=vocabulary_size, output_dim=25)),
model.add(GRU(1024, return_sequences=True)),
model.add(Dense(vocabulary_size))

In [55]:
for input_text,target_text in dataset.take(1):
    output=model.predict(input_text)
    print(output[0])

In [56]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [37]:
output[0]

NameError: name 'output' is not defined

In [152]:
si=tf.random.categorical(output[0],num_samples=1)
si

<tf.Tensor: shape=(29, 1), dtype=int64, numpy=
array([[39],
       [52],
       [38],
       [36],
       [51],
       [29],
       [ 6],
       [31],
       [37],
       [48],
       [ 7],
       [43],
       [25],
       [10],
       [31],
       [ 5],
       [11],
       [51],
       [ 6],
       [20],
       [44],
       [21],
       [30],
       [52],
       [41],
       [ 1],
       [47],
       [38],
       [19]])>

In [153]:
tf.squeeze(si,axis=1).numpy()

array([39, 52, 38, 36, 51, 29,  6, 31, 37, 48,  7, 43, 25, 10, 31,  5, 11,
       51,  6, 20, 44, 21, 30, 52, 41,  1, 47, 38, 19])

In [154]:
''.join(index2char[tf.squeeze(si,axis=1).numpy()])

'ویهمۀط\xa0عنژ«ّس؟ع|آۀ\xa0خْدظیُ\rچهح'

In [161]:
def loss_f(labels,logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels,logits,from_logits=True)
model.compile(optimizer='adam',loss=loss_f)

In [162]:
check_point=tf.keras.callbacks.ModelCheckpoint(filepath='chekponit_khayyam.weights.h5',save_weights_only=True)

In [163]:
history=model.fit(dataset,epochs=10,callbacks=[check_point])

Epoch 1/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 129ms/step - loss: 4.2698
Epoch 2/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 134ms/step - loss: 3.8802
Epoch 3/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 129ms/step - loss: 3.7390
Epoch 4/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 130ms/step - loss: 3.2374
Epoch 5/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 131ms/step - loss: 3.1167
Epoch 6/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 143ms/step - loss: 3.0410
Epoch 7/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 130ms/step - loss: 2.9357
Epoch 8/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 129ms/step - loss: 2.7697
Epoch 9/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 135ms/step - loss: 2.6219
Epoch 10/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 134ms/step - loss: 2.5574
